In [1]:
import pandas as pd
import re
import apache_beam as beam

print("Beam:", beam.__version__)
print("Pandas:", pd.__version__)


Failed to import GCSFileSystem; loading of this filesystem will be skipped. Error details: cannot import name 'storage' from 'google.cloud' (unknown location)


Beam: 2.74.0
Pandas: 2.2.3


In [2]:
import pandas as pd

print(pd.__version__)
print(hasattr(pd.DataFrame, "first"))

2.2.3
True


In [9]:
import pandas as pd

print(pd.__version__)
print(pd.DataFrame)
print(pd.DataFrame.__module__)
print(hasattr(pd.DataFrame, "first"))

3.0.3
<class 'pandas.DataFrame'>
pandas
False


In [3]:
import apache_beam as beam

with beam.Pipeline() as p:
    (
        p
        | beam.Create(["hello", "world"])
        | beam.Map(print)
    )

hello
world


In [4]:
df = pd.read_csv(r"food_daily.csv")
df.head()

,Customer_id,date,time,order_id,items,amount,mode,restaurnt,Status,ratings,feedback
0,JXJY167254JK,11/10/2023,8.31.21,654S654,PiZza:Marga?ritA:WATERZOOI:Crispy Onion Rings,21,Wallet,Brussels Mussels,Delivered,2,Late delivery
1,JXJY167254JK,11/10/2023,9.31.21,2444454,Noodles:Pizza:BREAD,97,Card,Saint German,Delivered,3,Stale food
2,XVTR474839TP,11/10/2023,4.31.31,397T397,Fried Rice:salaD,46,Card,Brussels Mussels,Delivered,3,Complicated procedure
3,UFDF355524DM,11/10/2023,5.31.21,428K428,noo%dles:,71,Card,Gaspar's,Delivered,1,Food not good
4,FRBT691245BA,11/10/2023,6.31.21,437M437,Soup of the day:,29,Online,Sushi Masters,Delivered,1,Stale food


In [19]:
df.columns

Index(['Customer_id', 'date', 'time', 'order_id', 'items', 'amount', 'mode',
       'restaurnt', 'Status', 'ratings', 'feedback'],
      dtype='str')

In [12]:
df.shape

(891, 11)

In [20]:
df.describe()

,amount,ratings
count,891.00000,891.000000
mean,60.91807,2.909091
std,31.96587,1.349838
min,12.00000,1.000000
25%,39.00000,2.000000
50%,59.00000,3.000000
75%,81.00000,4.000000
max,678.00000,5.000000


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Customer_id  891 non-null    str  
 1   date         891 non-null    str  
 2   time         891 non-null    str  
 3   order_id     891 non-null    str  
 4   items        891 non-null    str  
 5   amount       891 non-null    int64
 6   mode         891 non-null    str  
 7   restaurnt    891 non-null    str  
 8   Status       891 non-null    str  
 9   ratings      891 non-null    int64
 10  feedback     891 non-null    str  
dtypes: int64(2), str(9)
memory usage: 151.9 KB


In [5]:
def remove_last_colon(item):
    if item.endswith(":"):
        return item[:-1]
    return item
    
def process_row(row):
    columns = row.split(',')
    if len(columns) > 4:
        columns[4] = remove_last_colon(columns[4])
    return ','.join(columns)

def remove_special_characters(row):
    columns = row.split(',')
    ret = ''
    for col in columns:
        ret += re.sub(r'[^a-zA-Z0-9 ]', '', col) + ','
    return ret[:-1]

def print_row(row):
    print(row)



In [6]:
input_file = 'food_daily.csv'
output_path = 'outputs/processed'

with beam.Pipeline() as p:
    cleaned_data = (
        p
        | "Read Input File" >> beam.io.ReadFromText(input_file, skip_header_lines=1)
        | "Process Items Column" >> beam.Map(process_row)
        | "Convert to lowercase" >> beam.Map(lambda row: row.lower())
        | "Remove special characters" >> beam.Map(remove_special_characters)
    )

    delivered_orders = (
        cleaned_data
            | 'Filter delivered data' >> beam.Filter(lambda row: row and len(row.split(',')) > 8 and row.split(',')[8].strip().lower() == 'delivered')
            | 'Write Delivered File' >> beam.io.WriteToText(output_path + '/delivered', file_name_suffix='.csv')
        )

    undelivered_orders = (
        cleaned_data
            | 'Filter undelivered data' >> beam.Filter(lambda row: row and len(row.split(',')) > 8 and row.split(',')[8].strip().lower() != 'delivered')
            | 'Write Undelivered File' >> beam.io.WriteToText(output_path + '/undelivered', file_name_suffix='.csv')
        )